In [1]:
# ¿Qué segmentos, regiones y categorías concentran las ventas y la utilidad de Superstore?

from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)
DATA = ROOT / "data"
DETAIL = ROOT / "submission" / "superstore_enriched_sales.csv"
ANSWER = ROOT / "submission" / "sales_by_segment_region_category.csv"

In [2]:
# Las claves contextuales preservan las combinaciones de atributos que Order ID no identifica por sí solo.

orders = pd.read_csv(DATA / "orders.csv")
customers = pd.read_csv(DATA / "customers.csv")
products = pd.read_csv(DATA / "products.csv")
order_lines = pd.read_csv(DATA / "order_lines.csv")

orders.shape, customers.shape, products.shape, order_lines.shape

((1857, 7), (1191, 9), (926, 6), (1952, 9))

In [3]:
# Cada integración conserva una línea de orden y añade el contexto necesario para analizarla.

enriched_sales = (
    order_lines.merge(orders, on="order_context_key", validate="many_to_one")
    .merge(customers, on="customer_context_key", validate="many_to_one")
    .merge(products, on="product_context_key", validate="many_to_one")
)

enriched_sales.shape

(1952, 28)

In [4]:
# La respuesta se agrega después de integrar las fuentes, no antes.

sales_by_context = (
    enriched_sales.groupby(
        ["Customer Segment", "Region", "Product Category"], as_index=False
    )
    .agg(sales=("Sales", "sum"), profit=("Profit", "sum"), order_lines=("Sales", "size"))
    .sort_values(["sales", "profit"], ascending=False)
)

sales_by_context.head()

,Customer Segment,Region,Product Category,sales,profit,order_lines
40,Small Business,East,Office Supplies,102471.89,18035.766500,79
23,Corporate,West,Technology,100698.30,7594.468656,40
14,Corporate,Central,Technology,83461.18,16587.381500,51
15,Corporate,East,Furniture,70510.73,-3052.511512,30
21,Corporate,West,Furniture,69068.34,9387.075800,35


In [5]:
# La entrega conserva tanto el detalle integrado como la respuesta agregada para auditoría posterior.

assert len(enriched_sales) == len(order_lines)
assert enriched_sales[["Sales", "Profit", "Customer Segment", "Region", "Product Category"]].notna().all().all()

enriched_sales.to_csv(DETAIL, index=False)
sales_by_context.to_csv(ANSWER, index=False)